# Week 6 Assignment — Denoising Autoencoder on MNIST


**Pipeline:**
1. Load and preprocess MNIST
2. Add artificial (Gaussian) noise to create noisy inputs
3. Build and train a Denoising Autoencoder (noisy → clean)
4. Generate denoised outputs on the test set
5. Visualize original vs noisy vs reconstructed images
6. Discuss results, challenges, and key observations


## 1. Imports and Setup

In [ ]:
# YOUR CODE HERE (imports)
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)


## 2. Data Preparation

Load MNIST via `tf.keras.datasets.mnist` (equivalent to the Kaggle MNIST dataset referenced in the assignment resources).
We normalize pixel values to the `[0, 1]` range and reshape each image to `(28, 28, 1)` so it can be fed into `Conv2D` layers.


In [ ]:
# YOUR CODE HERE (load + preprocess)
(x_train, _), (x_test, _) = tf.keras.datasets.mnist.load_data()

# Normalize to [0, 1]
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Reshape to (28, 28, 1) for Conv2D layers
x_train = x_train.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)

print("Train shape:", x_train.shape)
print("Test shape:", x_test.shape)


## 3. Add Artificial Noise

We inject Gaussian noise into the images and clip pixel values back to `[0, 1]`.
The **noisy images are the model input**, and the **original clean images are the training target** —
this is what makes it a *denoising* autoencoder rather than a plain autoencoder.


In [ ]:
# YOUR CODE HERE (noise injection)
NOISE_FACTOR = 0.4

def add_noise(images, noise_factor=NOISE_FACTOR):
    noisy = images + noise_factor * np.random.normal(loc=0.0, scale=1.0, size=images.shape)
    return np.clip(noisy, 0.0, 1.0)

x_train_noisy = add_noise(x_train)
x_test_noisy = add_noise(x_test)

print("Noisy train shape:", x_train_noisy.shape)


In [ ]:
# Quick sanity check: original vs noisy
n = 5
plt.figure(figsize=(10, 4))
for i in range(n):
    ax = plt.subplot(2, n, i + 1)
    plt.imshow(x_train[i].squeeze(), cmap='gray')
    plt.title("Original")
    plt.axis('off')

    ax = plt.subplot(2, n, i + 1 + n)
    plt.imshow(x_train_noisy[i].squeeze(), cmap='gray')
    plt.title("Noisy")
    plt.axis('off')
plt.tight_layout()
plt.show()


## 4. Build the Denoising Autoencoder

**Encoder:** stacked `Conv2D` + `MaxPooling2D` layers compress the 28×28×1 image into a small
spatial latent representation.

**Decoder:** stacked `Conv2DTranspose` (or `UpSampling2D` + `Conv2D`) layers reconstruct the
28×28×1 image from that latent representation.

The final layer uses a `sigmoid` activation since pixel values are normalized to `[0, 1]`.


In [ ]:
# YOUR CODE HERE (model architecture)
def build_autoencoder():
    inputs = layers.Input(shape=(28, 28, 1))

    # Encoder
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    x = layers.MaxPooling2D((2, 2), padding='same')(x)          # 14x14x32
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)
    encoded = layers.MaxPooling2D((2, 2), padding='same')(x)     # 7x7x32

    # Decoder
    x = layers.Conv2DTranspose(32, (3, 3), strides=2, activation='relu', padding='same')(encoded)  # 14x14x32
    x = layers.Conv2DTranspose(32, (3, 3), strides=2, activation='relu', padding='same')(x)          # 28x28x32
    decoded = layers.Conv2D(1, (3, 3), activation='sigmoid', padding='same')(x)                       # 28x28x1

    autoencoder = models.Model(inputs, decoded, name="denoising_autoencoder")
    return autoencoder

autoencoder = build_autoencoder()
autoencoder.summary()


## 5. Compile and Train

We use the **Adam** optimizer and **binary cross-entropy** loss (standard choice for pixel values
in `[0, 1]`; MSE also works fine here). `EarlyStopping` guards against overfitting.


In [ ]:
# YOUR CODE HERE (compile + train)
autoencoder.compile(optimizer='adam', loss='binary_crossentropy')

early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = autoencoder.fit(
    x_train_noisy, x_train,
    epochs=20,
    batch_size=128,
    shuffle=True,
    validation_data=(x_test_noisy, x_test),
    callbacks=[early_stop]
)


In [ ]:
# Training curve
plt.figure(figsize=(6, 4))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Autoencoder Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Binary Cross-Entropy Loss')
plt.legend()
plt.show()


## 6. Generate Denoised Outputs on the Test Set

In [ ]:
# YOUR CODE HERE (inference)
denoised_images = autoencoder.predict(x_test_noisy)
print("Denoised output shape:", denoised_images.shape)


## 7. Result Visualization

Side-by-side comparison of **original**, **noisy**, and **reconstructed (denoised)** images for a
sample of test digits.


In [ ]:
n = 10
plt.figure(figsize=(20, 6))
for i in range(n):
    # Original
    ax = plt.subplot(3, n, i + 1)
    plt.imshow(x_test[i].squeeze(), cmap='gray')
    if i == 0:
        ax.set_ylabel("Original", fontsize=12)
    plt.xticks([]); plt.yticks([])

    # Noisy
    ax = plt.subplot(3, n, i + 1 + n)
    plt.imshow(x_test_noisy[i].squeeze(), cmap='gray')
    if i == 0:
        ax.set_ylabel("Noisy", fontsize=12)
    plt.xticks([]); plt.yticks([])

    # Denoised
    ax = plt.subplot(3, n, i + 1 + 2 * n)
    plt.imshow(denoised_images[i].squeeze(), cmap='gray')
    if i == 0:
        ax.set_ylabel("Denoised", fontsize=12)
    plt.xticks([]); plt.yticks([])

plt.suptitle("Original vs Noisy vs Denoised (Reconstructed)", fontsize=14)
plt.tight_layout()
plt.show()


## 8. Quantitative Evaluation (Optional / Innovation)

Beyond visual inspection, we can measure reconstruction quality with a per-image error metric,
e.g. **Mean Squared Error (MSE)** between the denoised output and the true clean image, and
compare it to the MSE between the noisy input and the clean image (to show improvement).


In [ ]:
# YOUR CODE HERE (quantitative comparison)
mse_noisy = np.mean((x_test_noisy - x_test) ** 2)
mse_denoised = np.mean((denoised_images - x_test) ** 2)

print(f"MSE (noisy vs clean):     {mse_noisy:.5f}")
print(f"MSE (denoised vs clean):  {mse_denoised:.5f}")
print(f"Improvement:              {(1 - mse_denoised / mse_noisy) * 100:.2f}% reduction in error")


## 9. Analysis and Observations


- **Data Preparation:** MNIST images were normalized to `[0, 1]` and reshaped to `(28, 28, 1)`.
  Gaussian noise (`noise_factor = 0.4`) was added and clipped to keep valid pixel range.
- **Model Architecture:** The encoder uses two `Conv2D + MaxPooling2D` blocks to compress the
  image into a `7x7x32` latent representation; the decoder mirrors this with `Conv2DTranspose`
  layers to reconstruct the original resolution.
- **Training Setup:** Adam optimizer, binary cross-entropy loss, `EarlyStopping` on validation
  loss to avoid overfitting.
- **Denoising Performance:** *(fill in — comment on how well digit structure is preserved, which
  digits are hardest to denoise, etc.)*
- **Challenges:** *(fill in — e.g. sensitivity to noise factor, blurring of fine strokes, choice
  of loss function)*
- **Key Observations:** *(fill in — e.g. MSE improvement %, visual quality trade-offs)*
- **Possible Extensions (Innovation):** try different noise types (salt-and-pepper), a deeper
  architecture, denoising with skip connections (U-Net style), or comparing against a
  fully-connected (dense) autoencoder baseline.
